# 记忆保存

目前，我们的智能体能记住对话，也会压缩记忆了。但它的记忆只存在于程序运行的时候：程序一关，或者 Jupyter 一重启，所有记忆就都没了。

我们希望智能体在重新启动以后，依然能够找回之前的记忆。最简单的方法就是及时把智能体的记忆保存到文件系统：

1. **保存**：每聊完一轮，就把记忆保存到文件里；
2. **编号**：每段记忆有一个自己的编号（ID），用来区分不同的**对话**（Conversation）；
3. **切换**：输入 `/new` 开始一段新的记忆，输入 `#` 加编号，加载以前保存的记忆。

把程序运行时的数据保存下来，以后还能加载回来，这叫作**持久化**（persistence）。

In [1]:
import json
import os
import uuid
from abc import ABC, abstractmethod
from enum import Enum
from pathlib import Path
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 消息类

### 角色

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

### 抽象类

我们需要在消息类增加一个功能，可以把内存中的消息对象转换成 JSON 格式，这样才可以保存到文件中。

* `to_dict`：新函数，返回消息的角色、文字内容的 JSON 格式;
*  `from_dict`：新函数，反过来，把从文件里读出来的 JSON 格式数据，变回消息类可以处理的消息对象。它前面的 `@classmethod` 表示这是一个**类方法**。

把对象变成可以保存的格式，叫作**序列化**；反过来，叫作**反序列化**。

一条消息保存成 JSON，大概是这个样子：

```json
{"type": "User", "content": "你好，我叫小明，正在学第 2 部分。"}
```

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    @classmethod
    def from_dict(cls, data: dict) -> Message:
        classes = {
            MsgType.SYSTEM.value: SystemMessage,
            MsgType.USER.value: UserMessage,
            MsgType.ASSISTANT.value: AssistantMessage,
        }
        return classes[data["type"]](data["content"])

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

### 系统消息类

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

### 用户消息类

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

### 助手消息类

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [7]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 记忆类

我们同样要在记忆类中增加序列化和反序列化函数，来辅助实现记忆持久化：

- `id`：记忆的编号。`uuid.uuid4()` 会生成一个几乎不可能重复的随机编号，我们只取前 8 个字符构成一个短编号，比如 `3f9a1c2e`。加载以前的记忆时，就用这个编号；
- `to_dict`：把整段记忆转换成 JSON 格式，包括编号和所有消息；
- `from_dict`：反过来，把加载的 JSON 格式数据重建一段记忆。它前面的 `@classmethod` 表示这是一个**类方法**。

In [8]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, message: Message):
        self.messages.append(message)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        self.note("summary", self.history())

    def token_count(self):
        return sum(len(msg.summary() or "") for msg in self.messages)

    @staticmethod
    def note(label: str, text: str):
        lines = str(text).splitlines() or [""]
        print(f"  * {label}:")
        for line in lines:
            print(f"  * {line}")

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [Message.from_dict(m) for m in data["messages"]]
        return memory

## 记忆仓库类

**记忆仓库类**（MemoryStore）负责把记忆存进文件、或者从文件里读出来。每段记忆存成一个文件，存放在 `memories` 文件夹里，文件名就是记忆的编号，比如 `memories/3f9a1c2e.json`。

- `__init__`：初始化，如果 `memories` 文件夹不存在，就创建它；
- `save`：把记忆转换成 JSON 格式，再用 `json.dumps` 转换为 JSON 格式字符串，这样就可以保存到文件中；
- `load`：按编号找到文件，读出 JSON 数据，再用 `Memory.from_dict` 还原成记忆。

In [9]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

## 大模型调用器

### 基础类

In [10]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

### OpenAI 调用器

In [11]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

### Claude 调用器

In [12]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## 智能体类

真正的以及持久化发生在智能体类内：

1. 多了一个 `self.store`，也就是记忆仓库；
2. 每聊完一轮，就调用 `self.store.save` 自动保存。这样即使程序突然关掉，最多也只会丢失正在进行的这一轮；
3. `chat` 多了两个命令：
   - `/new`：创建一段新的记忆，之前的记忆已经保存在文件里了，不会丢失；
   - `#` 编号：从仓库里加载这段记忆。

In [13]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None
        self.memory.note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                self.memory.note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    self.memory.note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    self.memory.note(f"loaded memory #{memory.id}", memory.history())
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            self.store.save(self.memory)
            print(f"AI: {response.summary()}")

## 大模型调用

先聊一句，让学习助手记住小明。这段记忆会被自动保存下来：

In [14]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学第 2 部分。"])
saved_id = agent.memory.id

  * memory:
  * #dfb75bdb

YOU: 你好，我叫小明，正在学第 2 部分。
AI: 你好，小明，欢迎来到《从大模型到智能体》学习之旅！

第 2 部分通常会开始更系统地理解：
- 大模型能做什么
- 智能体和普通聊天机器人的区别
- 任务拆解、规划、记忆、工具使用这些核心能力

如果你愿意，我可以用“新手友好”的方式陪你学第 2 部分，比如：
1. 先给你讲这一部分的核心概念  
2. 按章节帮你做笔记  
3. 用简单例子解释智能体怎么工作  
4. 出几道小练习帮你巩固

你可以直接告诉我：
- 你正在看的章节标题，或者
- 你想先弄懂哪个概念

我就从那里开始。

YOU: /quit


现在，`memories` 文件夹里已经多了一个文件。打开看看：

In [15]:
print(Path(f"memories/{saved_id}.json").read_text(encoding="utf-8"))

{
  "id": "dfb75bdb",
  "messages": [
    {
      "type": "User",
      "content": "你好，我叫小明，正在学第 2 部分。"
    },
    {
      "type": "Assistant",
      "content": "你好，小明，欢迎来到《从大模型到智能体》学习之旅！\n\n第 2 部分通常会开始更系统地理解：\n- 大模型能做什么\n- 智能体和普通聊天机器人的区别\n- 任务拆解、规划、记忆、工具使用这些核心能力\n\n如果你愿意，我可以用“新手友好”的方式陪你学第 2 部分，比如：\n1. 先给你讲这一部分的核心概念  \n2. 按章节帮你做笔记  \n3. 用简单例子解释智能体怎么工作  \n4. 出几道小练习帮你巩固\n\n你可以直接告诉我：\n- 你正在看的章节标题，或者\n- 你想先弄懂哪个概念\n\n我就从那里开始。"
    }
  ]
}


接下来：用 `/new` 开始一段新记忆，问智能体：“我叫什么名字？”，它应该答不出来；然后加载刚才保存的记忆，再问一遍：

In [16]:
# start a new memory, then load the saved one again
agent.chat(["/new", "我叫什么名字？", f"#{saved_id}", "我叫什么名字？"])
# or call agent.chat() without questions to type your own

  * memory:
  * #dfb75bdb

YOU: /new
  * new memory:
  * #49c81eb5

YOU: 我叫什么名字？
AI: 我不知道你的名字，除非你告诉我。  
如果你愿意，可以直接告诉我，我之后就能叫你这个名字。

YOU: #dfb75bdb
  * loaded memory #dfb75bdb:
  * [User]: 你好，我叫小明，正在学第 2 部分。
  * [Assistant]: 你好，小明，欢迎来到《从大模型到智能体》学习之旅！
  * 
  * 第 2 部分通常会开始更系统地理解：
  * - 大模型能做什么
  * - 智能体和普通聊天机器人的区别
  * - 任务拆解、规划、记忆、工具使用这些核心能力
  * 
  * 如果你愿意，我可以用“新手友好”的方式陪你学第 2 部分，比如：
  * 1. 先给你讲这一部分的核心概念  
  * 2. 按章节帮你做笔记  
  * 3. 用简单例子解释智能体怎么工作  
  * 4. 出几道小练习帮你巩固
  * 
  * 你可以直接告诉我：
  * - 你正在看的章节标题，或者
  * - 你想先弄懂哪个概念
  * 
  * 我就从那里开始。

YOU: 我叫什么名字？
  * summary:
  * [User]: Historical Summary: 用户自我介绍为“小明”，正在学习《从大模型到智能体》的第 2 部分；随后询问自己的名字。
  * [Assistant]: 你叫小明。
AI: 你叫小明。

YOU: /quit


加载之后，智能体恢复了记忆，又记起了小明。即使重启 Jupyter，只要记得编号，也能把这段对话找回来。

> 💡 聊天记录是以普通文本保存在电脑上的，任何能打开这个文件夹的人都能看到。真实的应用里，要注意保护好这些数据。

## 动手试试

- 重启 Jupyter，重新运行代码，然后用 `#` 加编号加载之前的记忆，验证它真的被保存下来了。
- 给 `chat` 加一个 `/list` 命令，列出所有保存过的记忆编号。提示：给 `MemoryStore` 加一个方法，用 `self.root.glob("*.json")` 找到所有文件，文件的 `stem` 属性就是去掉 `.json` 后的文件名。
- 第 4 章的练习里，我们把学习助手做成了命令行程序。把这一章的代码也放进去，试试关掉程序、重新打开、再接着聊。

## 小结

- 把程序运行时的数据保存下来，下次还能用，叫作**持久化**；
- 我们用 **JSON** 格式保存记忆：`to_dict` 把消息和记忆转换成 JSON 格式（序列化），`from_dict` 再把 JSON 数据转换回对象（反序列化）；
- 每段记忆有一个随机生成的**编号**，`MemoryStore` 把它保存为 `memories/编号.json`；
- `Agent` 每轮自动保存，并支持 `/new` 开始新记忆、`#编号` 加载旧记忆。